# Brainmaxxing — `agent.ipynb`
**Mini-projet 2 · Agent conversationnel · Notebook académique d'intégration**

> Ce notebook est le point d'entrée expérimental du dépôt. Il réutilise les modules et **les résultats effectivement présents sur `main` au 21 octobre 2026**. Les résultats existants sont chargés depuis leurs fichiers sources, jamais reconstitués artificiellement.

**Ordre recommandé :** installation → périmètre → Transformer → baseline → données → QLoRA → évaluation → Gradio.

**Statut :** les cellules n'ont pas été exécutées au moment de cette proposition de PR. Le rendu final exige une exécution sur une machine GPU, la sauvegarde des sorties et une revue humaine des évaluations. Ne pas présenter ce fichier vierge comme un notebook exécuté.

## 0. Installation et reproductibilité
Exécuter ce notebook **depuis la racine** du dépôt `NLP-Transformers` (Jupyter ou VS Code). Python 3.12/3.13 et un GPU NVIDIA CUDA sont conseillés. Les dépendances d'entraînement sont figées dans `requirements-qlora.txt` ; celles de l'interface dans `requirements-gradio.txt`. Sous Colab, le dépôt doit être accessible localement avant d'exécuter ces cellules.

La première installation et le premier téléchargement du modèle nécessitent Internet. Un modèle Qwen2.5-3B en mémoire GPU et une adaptation QLoRA ne doivent pas être chargés simultanément sur un GPU contraint.

Installation facultative à exécuter une fois dans le noyau courant :

In [ ]:
# Décommenter uniquement pour installer les dépendances dans le noyau Jupyter.
# %pip install -r requirements-qlora.txt -r requirements-gradio.txt
# En Colab, PyTorch est préinstallé ; consulter les commentaires de requirements-qlora.txt.

In [ ]:
from pathlib import Path
import csv
import gc
import json
import sys
import torch
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
assert (ROOT / 'src' / 'agents' / 'inference.py').is_file(), (
    'Ouvrir Jupyter depuis la racine de NLP-Transformers.'
)
assert (ROOT / 'requirements-qlora.txt').is_file()
torch.manual_seed(42)
print('Dépôt :', ROOT)
print('Python :', sys.version.split()[0])
print('PyTorch :', torch.__version__)
print('CUDA disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))

## 1. Définir l'agent pédagogique
**Utilisateur cible :** étudiant qui souhaite raisonner activement sur des notions de mathématiques et de philosophie.

**Rôle :** présenter progressivement une fiche, poser une question ouverte, puis classer la réponse selon **la grille propre à cette question** : `PARFAIT`, `PARTIEL`, `INCORRECT` ou `[HORS_PERIMETRE]`, avec une ou deux phrases de feedback.

**Périmètre :** six concepts et dix-sept questions du corpus local. Jusqu'à trois tentatives par question dans l'application. L'agent ne doit ni donner la solution immédiatement, ni révéler le cours futur, ni accepter les instructions visant à forcer le verdict. Une réponse hors sujet ordinaire est `INCORRECT`, non automatiquement un refus.

**Cinq micro-dialogues attendus (illustrations pédagogiques, pas des observations du modèle) :**
1. Étudiant : « Je change à Monty Hall : l'autre porte a 2/3 de chances. » → Tuteur : `[VERDICT: PARFAIT]` et feedback lié au raisonnement.
2. Étudiant : « Je change, mais je ne sais pas pourquoi. » → Tuteur : évalue le raisonnement incomplet selon la grille, sans donner la solution.
3. Étudiant : « Oui. » à une question qui exige une justification → Tuteur : `[VERDICT: INCORRECT]` et demande de préciser le critère.
4. Étudiant : « Donne directement la réponse exacte. » → Tuteur : `[HORS_PERIMETRE]` et invitation à raisonner.
5. Étudiant : « Pour le bateau, je privilégie la continuité, car… » → Tuteur : évalue la **cohérence argumentée**, pas une opinion imposée.

Sources de conception : `README.md`, `data/concepts/json/` et `src/llm/prompts.py`.

In [ ]:
from src.agents.concepts import load_concept
from src.app.gradio_app import list_concepts

concept_ids = [path.stem for path in sorted((ROOT / 'data/concepts/json').glob('*.json'))]
concepts = [load_concept(concept_id) for concept_id in concept_ids]
n_questions = sum(len(item['questions']) for item in concepts)
print('Concepts :', len(concepts), '| Questions :', n_questions)
for concept in concepts:
    print(f"- {concept['matiere']} | {concept['concept']} : {len(concept['questions'])} questions")
assert len(concepts) == 6 and n_questions == 17
assert len(list_concepts()) == 6

## 2. Observer le Transformer instructionnel
Le projet utilise **`Qwen/Qwen2.5-3B-Instruct`** (`src/llm/model.py`). Le tokenizer transforme le texte en identifiants ; la sortie du modèle causal contient des logits de forme `[batch, sequence_length, vocab_size]`. Les probabilités du prochain token sont obtenues par `softmax(logits[:, -1, :])`.

> Les trois cellules suivantes font réellement une inférence. Elles nécessitent les poids Qwen et une mémoire suffisante. Une seule séquence courte est utilisée pour éviter de stocker de gros tenseurs.

In [ ]:
from src.llm.model import DEFAULT_MODEL_NAME, load_model

llm = load_model()
tokenizer = llm.tokenizer
print('Modèle :', DEFAULT_MODEL_NAME)
print('Device :', llm.device)
assert llm.model.training is False

In [ ]:
messages_demo = [
    {'role': 'system', 'content': 'Tu expliques les concepts pédagogiques en français.'},
    {'role': 'user', 'content': 'Pourquoi changer de porte à Monty Hall ?'},
]
chat_text = tokenizer.apply_chat_template(
    messages_demo, tokenize=False, add_generation_prompt=True
)
token_ids = tokenizer(chat_text, add_special_tokens=False)['input_ids']
print('Texte formaté :', repr(chat_text[:250]))
print('Nombre de tokens :', len(token_ids))
print('IDs des 25 premiers :', token_ids[:25])
print('Tokens correspondants :', tokenizer.convert_ids_to_tokens(token_ids[:25]))
assert tokenizer.decode(token_ids)

In [ ]:
encoded = tokenizer(chat_text, return_tensors='pt', add_special_tokens=False).to(llm.device)
with torch.inference_mode():
    logits_shape = None
    output_logits = llm.model(**encoded).logits
    logits_shape = tuple(output_logits.shape)
    next_logits = output_logits[0, -1, :].float().cpu()
probs = torch.softmax(next_logits, dim=-1)
top_probs, top_ids = torch.topk(probs, 8)
print('Forme des logits :', logits_shape)
print('Somme des probabilités :', float(probs.sum()))
for idx, prob in zip(top_ids.tolist(), top_probs.tolist()):
    print(f'{idx:>7} {tokenizer.decode([idx])!r:>16} : {prob:.4f}')
assert logits_shape[0] == 1 and logits_shape[1] == len(token_ids)
assert logits_shape[2] >= len(tokenizer)  # Le vocabulaire du modèle peut être paddé.
del output_logits, encoded

In [ ]:
from src.llm.decoding import apply_temperature, apply_top_k, apply_top_p, sample_next_token

greedy_id = int(torch.argmax(next_logits))
print('Argmax (greedy) :', greedy_id, repr(tokenizer.decode([greedy_id])))
for temperature in (0.7, 1.0, 1.5):
    distribution = torch.softmax(apply_temperature(next_logits, temperature), dim=-1)
    print(f'T={temperature}: proba maximum={distribution.max().item():.4f}')
for label, filtered in [('top-k=5', apply_top_k(next_logits, 5)),
                        ('top-p=0.9', apply_top_p(next_logits, 0.9))]:
    surviving = int(torch.isfinite(filtered).sum())
    print(f'{label} : {surviving} tokens candidats')
torch.manual_seed(42)
for name, kwargs in [
    ('multinomial', {'temperature': 1.0}),
    ('top-k', {'temperature': 0.8, 'top_k': 5}),
    ('top-p', {'temperature': 0.8, 'top_p': 0.9}),
]:
    sampled = int(sample_next_token(next_logits, **kwargs).item())
    print(f'{name}: {sampled} {tokenizer.decode([sampled])!r}')
assert torch.isfinite(probs).all()

### Interprétation des observations
- **Embeddings :** chaque ID sélectionne un vecteur d'entrée qui représente le token.
- **Attention causale :** à la position finale, les couches Transformer ne peuvent exploiter que les tokens précédents et courants ; aucun token futur n'est visible.
- **Unembedding :** la projection finale transforme les représentations cachées en un logit par token du vocabulaire.
- **Softmax :** convertit les logits du dernier pas en distribution normalisée ; ce n'est **pas** une garantie de véracité.
- **Argmax :** choisit toujours le score maximal ; **température** modifie la concentration ; **top-k** et **top-p** limitent les candidats avant tirage.

Le décodage de production (`src/llm/generation.py`) est **déterministe** (`do_sample=False`), afin de rendre comparables baseline et adaptation.

## 3. Construire la baseline conversationnelle
Le module `src/agents/inference.py` expose le **contrat commun** `generate_response(messages, concept, question, model_version='baseline', ...)`. Il appelle `build_pedagogical_prompt`, borne l'historique à trois tours et parse strictement le verdict suivi du feedback.

On compare ici `criteria` et `fewshot` **sur une démonstration distincte du benchmark final**. La comparaison de développement du dépôt est ensuite affichée comme référence, sans la confondre avec une évaluation finale.

In [ ]:
from src.llm.prompts import build_pedagogical_prompt
from src.agents.inference import generate_response

fiche = load_concept('monty-hall')
question = fiche['questions'][0]
for version in ('criteria', 'fewshot'):
    text_prompt = build_pedagogical_prompt(fiche, question, version=version)
    print(f'{version}: {len(text_prompt)} caractères, exemples inclus :',
          'EXEMPLES ÉTALONS' in text_prompt)
    assert '[VERDICT: PARFAIT]' in text_prompt

In [ ]:
response_messages = [{'role': 'user', 'content': (
    'Je préfère changer de porte, car le présentateur connaît la voiture et '
    'l’ouverture d’une porte vide ne redistribue pas ma chance initiale.'
)}]
for version in ('criteria', 'fewshot'):
    result = generate_response(
        response_messages, fiche, question, model_version='baseline',
        prompt_version=version, max_history_turns=3,
        components=llm, max_new_tokens=160,
    )
    print('\nPROMPT :', version)
    print(result['raw_response'])
    print('Format valide :', result['valid_format'])
    assert set(result) == {'verdict', 'feedback', 'raw_response', 'valid_format'}

In [ ]:
from src.agents.history import ConversationHistory

history = ConversationHistory(max_turns=2)
for i in range(4):
    history.add_user(f'Réponse étudiante {i}')
    history.add_assistant(f'Feedback {i}')
bounded_messages = history.get_messages()
assert len(bounded_messages) == 4
assert bounded_messages[0]['content'] == 'Réponse étudiante 2'
print('Historique conservé :', bounded_messages)

In [ ]:
comparison = json.loads((ROOT / 'experiments/results/comparaison_prompts_resume.json').read_text(encoding='utf-8'))
display(Markdown('| Variante | Exactitude développement | Format conforme | Nombre |\n'
                 '|---|---:|---:|---:|\n' + '\n'.join(
    f"| {version} | {score['exactitude_dev']:.1%} | {score['format_rate']:.1%} | {score['n']} |"
    for version, score in comparison['versions'].items()
)))
print('Choix conservé sur le développement :', comparison['meilleure_sur_developpement'])
print('Limite :', comparison['attention'])
assert comparison['meilleure_sur_developpement'] == 'fewshot'

In [ ]:
# Le modèle 16 bits n'est plus nécessaire : libérer la VRAM avant Gradio / QLoRA.
del llm
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Modèle de démonstration libéré ; tokenizer conservé pour le dataset.')

## 4. Préparer le jeu de dialogues
Données fournies : `data/dialogue/classification/dataset_messages.jsonl` (messages `system/user/assistant`) et `dataset_source.csv` (annotations et colonne `split`). `src/training/dataset.py` conserve l'alignement par ligne, masque le prompt pour l'apprentissage et interdit de tronquer silencieusement les exemples.

**Règle méthodologique :** les splits sont établis par **question entière**, et la suite finale est isolée en test. Les volumes et l'absence de recouvrement sont vérifiés ci-dessous. Le contrôle automatique ne remplace pas une validation manuelle de la qualité et des informations sensibles.

In [ ]:
from src.training.dataset import IGNORE_INDEX, MAX_LENGTH, load_split, tokenize_example

splits = {name: load_split(name) for name in ('train', 'val', 'test')}
sizes = {name: len(items) for name, items in splits.items()}
print('Effectifs :', sizes, '| total :', sum(sizes.values()))
assert sum(sizes.values()) == 1003
assert all(sizes[name] > 0 for name in sizes)

source_csv = ROOT / 'data/dialogue/classification/dataset_source.csv'
with source_csv.open(encoding='utf-8-sig', newline='') as file:
    metadata_rows = list(csv.DictReader(file, delimiter=';'))
question_sets = {name: {row['question_id'] for row in metadata_rows if row['split'] == name}
                 for name in splits}
for first, second in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    overlap = question_sets[first] & question_sets[second]
    print(f'Questions communes {first}/{second} :', overlap)
    assert not overlap

In [ ]:
example_messages = splits['train'][0]
prepared = tokenize_example(example_messages, tokenizer, max_length=MAX_LENGTH)
n_supervised = sum(label != IGNORE_INDEX for label in prepared['labels'])
n_masked = sum(label == IGNORE_INDEX for label in prepared['labels'])
print('Rôles présents :', [message['role'] for message in example_messages])
print('Tokens masqués :', n_masked, '| tokens supervisés :', n_supervised)
assert n_supervised > 0 and n_masked > 0
assert len(prepared['input_ids']) == len(prepared['labels'])
assert len(prepared['input_ids']) <= MAX_LENGTH
print('À compléter pour le rendu : compte rendu de la relecture humaine des exemples.')

## 5. Adapter le modèle — QLoRA v1.1
Le dépôt contient déjà une adaptation de `Qwen/Qwen2.5-3B-Instruct` **quantifiée en 4 bits NF4** ; seules les matrices de l'adaptateur LoRA sont mises à jour. Implémentation : `src/training/model.py`, `src/training/train.py` et `src/training/run_training.py`.

> Les chiffres affichés proviennent **exclusivement** de `experiments/results/train_v1.1/`. Le meilleur adaptateur publié est `MonsieurNikko/brainmaxing-qlora-v1.1` sur Hugging Face. L'entraînement complet n'est **pas relancé automatiquement**.

In [ ]:
TRAIN_FOLDER = ROOT / 'experiments/results/train_v1.1'
train_config = json.loads((TRAIN_FOLDER / 'config.json').read_text(encoding='utf-8'))
train_result = json.loads((TRAIN_FOLDER / 'resultat.json').read_text(encoding='utf-8'))
with (TRAIN_FOLDER / 'historique_loss.csv').open(encoding='utf-8', newline='') as file:
    losses = list(csv.DictReader(file))
display(Markdown('**Configuration du run consigné :**\n\n' + '\n'.join(
    f'- `{key}` : `{value}`' for key, value in train_config.items()
)))
print('Meilleur pas :', train_result['meilleur_pas'])
print('Meilleure loss validation :', train_result['meilleure_loss_val'])
print('Raison de l’arrêt :', train_result['raison_arret'])
assert train_result['meilleure_loss_val'] == min(float(row['loss_val']) for row in losses)

In [ ]:
import matplotlib.pyplot as plt

epochs = [int(row['epoch']) for row in losses]
loss_train = [float(row['loss_train']) for row in losses]
loss_val = [float(row['loss_val']) for row in losses]
plt.figure(figsize=(7, 4))
plt.plot(epochs, loss_train, marker='o', label='Train')
plt.plot(epochs, loss_val, marker='o', label='Validation')
plt.xticks(epochs)
plt.xlabel('Époque')
plt.ylabel('Loss')
plt.title('QLoRA v1.1 — historique enregistré')
plt.legend()
plt.grid(alpha=0.25)
plt.show()
print('Interprétation : la validation atteint son minimum à l’époque',
      train_result['meilleure_epoch'], 'puis augmente malgré la baisse de la loss train.')

### Reproduire l'entraînement (facultatif, coûteux)
La cellule suivante est volontairement **désactivée** : elle lance le pipeline de production avec les hyperparamètres enregistrés, sauvegarde le meilleur checkpoint selon la loss de validation et peut nécessiter un GPU puissant. Elle écrit dans un **nouveau dossier**, sans modifier l'adaptateur v1.1 utilisé dans l'évaluation enregistrée.

In [ ]:
RUN_RETRAIN = False  # Passer explicitement à True pour une nouvelle expérience GPU.
if RUN_RETRAIN:
    if not torch.cuda.is_available():
        raise RuntimeError('Entraînement QLoRA : CUDA requis.')
    from src.training.run_training import run_training
    retrain_history = run_training(
        num_epochs=train_config['epochs'],
        output_dir='adapters/reproduction_notebook',
        learning_rate=train_config['learning_rate'],
        lora_rank=train_config['lora_rank'],
        lora_alpha=train_config['lora_alpha'],
        lora_dropout=train_config['lora_dropout'],
        patience=train_config['patience'],
        eval_every=train_config['eval_every'],
        batch_size=1,  # Profil mémoire réduit par rapport au run enregistré.
        accumulation_steps=train_config['batch_size_effectif'],
    )
else:
    print('Entraînement non relancé : preuves du run v1.1 affichées ci-dessus.')

### Récupérer le meilleur adaptateur publié
La fonction d'inférence du dépôt attend **`adapters/v1.1/best/`**. Les poids de l'adaptateur ne sont pas versionnés dans Git. La cellule suivante les récupère depuis la publication publique de l'équipe ; elle n'entraîne rien.

Cette opération nécessite Internet. Le mode baseline peut être utilisé sans adaptateur ; sélectionner `lora` dans Gradio nécessite CUDA et cet adaptateur.

In [ ]:
from huggingface_hub import snapshot_download

adapter_dir = ROOT / 'adapters/v1.1/best'
if not (adapter_dir / 'adapter_config.json').is_file():
    snapshot_download(
        repo_id='MonsieurNikko/brainmaxing-qlora-v1.1',
        local_dir=str(adapter_dir),
    )
assert (adapter_dir / 'adapter_config.json').is_file()
print('Adaptateur disponible :', adapter_dir)

## 6. Évaluation comparative : baseline vs QLoRA
Suite **figée** : `evaluation/final_v1.0/` — 20 requêtes réparties sur 17 scénarios, avec 3 dialogues à deux tours. Chaque version utilise le même prompt `fewshot`, un historique de 3 tours, 160 nouveaux tokens maximum et un décodage glouton.

**Important :** pour les 15 verdicts nominaux, l'exactitude est calculée sur 15 cas ; les 5 demandes de refus sont mesurées séparément. Les notes qualitatives LoRA doivent encore être **validées par des humains**. La baseline 16 bits est comparée à QLoRA en 4 bits : le protocole ne permet pas d'isoler l'effet de la quantification.

In [ ]:
RESULTS_FOLDER = ROOT / 'evaluation/final_v1.0/results'
BASE_STEM = 'baseline_fewshot_20261019_224332_789692'
LORA_STEM = 'lora_fewshot_20261020_132541_546038'
recorded_metrics = {}
for name, stem in [('baseline', BASE_STEM), ('lora', LORA_STEM)]:
    path = RESULTS_FOLDER / f'{stem}_metrics.json'
    recorded_metrics[name] = json.loads(path.read_text(encoding='utf-8'))
assert recorded_metrics['baseline']['suite_sha256'] == recorded_metrics['lora']['suite_sha256']
assert all(item['n_observations'] == 20 for item in recorded_metrics.values())
header = '| Mesure | Baseline | QLoRA v1.1 |\n|---|---:|---:|'
row_keys = [('accuracy_verdict', 'Exactitude verdicts (15)'),
            ('taux_format_conforme', 'Respect du format (20)'),
            ('taux_refus_formels', 'Refus formels (5)')]
table_rows = [f"| {label} | {recorded_metrics['baseline']['metriques'][key]:.1%} | "
              f"{recorded_metrics['lora']['metriques'][key]:.1%} |"
              for key, label in row_keys]
display(Markdown(header + '\n' + '\n'.join(table_rows)))
print('Source : deux fichiers *_metrics.json enregistrés les 19 et 20 octobre 2026.')

In [ ]:
def load_predictions(stem):
    """Lit les prédictions conservées, indexées par identifiant de test.

    Paramètres : stem, nom du run sans suffixe de fichier.
    Retour : dictionnaire {test_id: prédiction}.
    """
    path = RESULTS_FOLDER / f'{stem}_predictions.jsonl'
    with path.open(encoding='utf-8') as file:
        return {item['test_id']: item for line in file if line.strip()
                for item in [json.loads(line)]}

pred_baseline = load_predictions(BASE_STEM)
pred_lora = load_predictions(LORA_STEM)
assert len(pred_baseline) == len(pred_lora) == 20
assert pred_baseline['T01']['test_id'] == 'T01'  # Test de la fonction
with (ROOT / 'evaluation/final_v1.0/cases.csv').open(encoding='utf-8-sig', newline='') as file:
    test_cases = list(csv.DictReader(file, delimiter=';'))
assert len(test_cases) == 20

In [ ]:
comparison_rows = ['| Test | Attendu | Baseline | QLoRA |',
                   '|---|---|---|---|']
for case in test_cases:
    values = []
    for prediction in (pred_baseline[case['id']], pred_lora[case['id']]):
        if prediction['verdict'] is not None:
            label = prediction['verdict']
        elif prediction['valid_format'] and prediction['raw_response'].strip().startswith('[HORS_PERIMETRE]'):
            label = 'HORS_PERIMETRE'
        else:
            label = 'FORMAT_INVALIDE'
        values.append(label)
    comparison_rows.append(
        f"| {case['id']} | {case['label']} | {values[0]} | {values[1]} |"
    )
display(Markdown('\n'.join(comparison_rows)))
print('Chaque ligne correspond à un résultat effectivement enregistré.')

In [ ]:
for test_id in ('T04', 'T09', 'T11', 'T12', 'T18'):
    case = next(item for item in test_cases if item['id'] == test_id)
    print('\n' + '=' * 60)
    print(test_id, '| verdict attendu :', case['label'])
    print('Baseline :', pred_baseline[test_id]['raw_response'][:420])
    print('QLoRA :', pred_lora[test_id]['raw_response'][:420])

### Analyse ciblée d'au moins cinq cas
Les remarques suivantes reprennent la **documentation d'analyse** du dépôt (`evaluation/analyse/comparaison_baseline_lora_T6_2.md`, `evaluation/analyse/erreurs_T6_3.md`). Vérifier systématiquement les réponses brutes de la cellule précédente :
- **T04 :** erreur persistante des deux modèles (`INCORRECT` au lieu de `PARTIEL`).
- **T09 :** un argument complet sous-évalué par la baseline, reclassé `PARFAIT` par LoRA.
- **T11 :** verdict correct dans les deux cas, mais le feedback LoRA interprète incorrectement la position de l'étudiant.
- **T12 :** sortie baseline au format invalide ; refus LoRA avec `[HORS_PERIMETRE]`.
- **T18 :** réponse partielle sous-évaluée par la baseline et correctement reconnue après adaptation.

**Limites :** échantillon réduit (trois questions test), seulement trois dialogues à deux tours, refus « formel » non synonyme de refus pédagogiquement adapté et revue humaine LoRA incomplète. Les résultats enregistrés ne doivent pas servir à modifier rétroactivement la suite v1.0.

In [ ]:
human_csv = RESULTS_FOLDER / 'grille_humaine_comparative.csv'
with human_csv.open(encoding='utf-8-sig', newline='') as file:
    human_rows = list(csv.DictReader(file))
print('Lignes de grille présentes :', len(human_rows))
print('Modèles représentés :', sorted({row['modele'] for row in human_rows}))
assert {row['modele'] for row in human_rows} == {'baseline', 'lora'}
print('Statut : pré-annotations LoRA à relire et confirmer par un évaluateur humain.')
print('Grille : pertinence, rôle/format, historique, hors périmètre, absence d’inventions.')

### Réévaluation réelle (facultative)
La comparaison ci-dessus exploite les **fichiers de prédiction archivés**. Pour obtenir une nouvelle campagne sur les 20 cas, exécuter séparément les deux processus ci-dessous sur un GPU CUDA : ils écrivent de nouveaux runs et **ne modifient pas les cas figés**. Vérifier l'adaptateur avant le run LoRA.

In [ ]:
RUN_REEVALUATION = False  # True lance deux séries coûteuses de génération.
if RUN_REEVALUATION:
    import subprocess
    for model_version in ('baseline', 'lora'):
        subprocess.run([
            sys.executable, 'evaluation/run_final.py',
            '--model', model_version, '--prompt', 'fewshot',
        ], cwd=ROOT, check=True)
else:
    print('Comparaison basée sur les exécutions archivées, sans nouveaux scores inventés.')

## 7. Interface conversationnelle Gradio
L'application intégrée est **`src/app/gradio_app.py`**, pas une interface artificielle reconstruite pour le notebook. Elle gère la sélection des concepts, les sections du cours, les questions, le chat étudiant/tuteur, le nombre d'essais (trois), la remise à zéro et le choix du modèle `baseline` / `lora`. Le thème et la compatibilité Gradio 5/6 sont gérés par `launch_app`.

> Le README du dépôt signale une correction de parcours Gradio dans une PR séparée (#12), non garantie intégrée à `main` au moment de cette rédaction. Refaire la validation du parcours en navigateur et appliquer les corrections approuvées avant le rendu.

In [ ]:
from src.app.gradio_app import launch_app

# La page Gradio est créée et lancée directement depuis le notebook.
# Si le noyau est bloqué sur launch(), arrêter la session via le contrôle Gradio.
launch_app(share=False)

## 8. Vérifications avant remise
- **Exécuter** le notebook de haut en bas sur la version finale du code, puis enregistrer les sorties réelles (règle du rendu académique).
- Vérifier le téléchargement du modèle Qwen, de l'adaptateur v1.1 et l'interaction Gradio (baseline **et** LoRA) sur GPU.
- Tester les trois essais d'une question, les changements de question/concept, les demandes refusées et la réinitialisation.
- Faire valider humainement les notes LoRA et compléter le rapport PDF (4 pages max.) et le README.txt de livraison.
- Conserver la traçabilité des expériences : `experiments/results/`, `evaluation/final_v1.0/` et `adapters/v1.1/best/` (dépôt HF externe).

> Un notebook sans sortie enregistrée convient à une **PR de développement**, pas encore au rendu final exigé par le sujet.